# 문제 2-1 : 콤마 구분 리스트 파서 활용

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0.7)

parser = CommaSeparatedListOutputParser()
print(parser.get_format_instructions())

pt = PromptTemplate(
    template="한국의 '{topic}' 분야와 관련된 유명한 장소나 활동 5가지를 추천해 주세요.\n"
             "각 항목은 '지역 + 이름'처럼 구체적으로 쓰고, 다른 설명은 덧붙이지 마세요.\n{fmt}",
    input_variables=["topic"],
    partial_variables={"fmt": parser.get_format_instructions()},
)

chain = pt | llm | parser

topic = input("관심 분야를 입력하세요 (예: 음식, 스포츠, 영화): ") or "음식"
try:
    r = chain.invoke({"topic": topic})
    print(type(r))
    print(r)
except Exception as e:
    print("오류:", e)


Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
<class 'list'>
['서울 남대문시장', '부산 자갈치시장', '전주 한옥마을', '제주 올레길', '경주 황남빵']
